# 04 - Fetch Calendar Features

Builds the calendar/holiday feature data used later alongside weather as
predictors: German **public holidays** for Nordrhein-Westfalen (NRW) and
**NRW school holidays** (Schulferien).

- Public holidays are computed algorithmically with the [`holidays`](https://pypi.org/project/holidays/) library - no network
  call, no staleness risk.
- School holidays are set per-year by the state ministry and are
  **not** rule-computable, so they are fetched once from the
  [OpenHolidays API](https://openholidaysapi.org/) and cached as a
  static CSV under `data/raw/calendar/`.

School-holiday data from OpenHolidays API is licensed **ODbL-1.0**;
attribution: data (c) OpenHolidays API contributors,
https://openholidaysapi.org/.

This notebook: determines the year range to cover from the bike-count
data already on disk, computes public holidays, fetches + saves school
holidays, and does a first-look sanity check on both.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Make `src/` importable regardless of whether this notebook is run
# from `notebooks/` (the normal case) or the project root.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from muenster_bike_forecast.data.calendar import (
    DEFAULT_PUBLIC_HOLIDAY_SUBDIV,
    DEFAULT_SUBDIVISION_CODE,
    fetch_school_holidays,
    public_holidays,
    save_school_holidays,
)

RAW_BIKE_COUNTS_DIR = PROJECT_ROOT / "data" / "raw" / "bike_counts"
CALENDAR_OUTPUT_DIR = PROJECT_ROOT / "data" / "raw" / "calendar"

## 1. Determine the year range to cover

Reads the min/max `datetime` across every station CSV already fetched
by `01_fetch_bike_counts.ipynb` (if present) to find the actual span of
bike-count data, then pads it: one year back for safety margin, and
forward to `2027` to cover a 24h-ahead forecast horizon into the near
future. Falls back to a sensible fixed range if no bike-count data is
on disk yet (e.g. a fresh checkout).

In [2]:
FALLBACK_START_YEAR = 2019
FALLBACK_END_YEAR = 2027

station_csvs = (
    [p for p in RAW_BIKE_COUNTS_DIR.glob("*.csv") if p.stem.isdigit()]
    if RAW_BIKE_COUNTS_DIR.exists()
    else []
)

if station_csvs:
    span_min, span_max = None, None
    for path in station_csvs:
        dt = pd.to_datetime(pd.read_csv(path, usecols=["datetime"])["datetime"])
        span_min = dt.min() if span_min is None else min(span_min, dt.min())
        span_max = dt.max() if span_max is None else max(span_max, dt.max())
    START_YEAR = min(FALLBACK_START_YEAR, span_min.year - 1)
    END_YEAR = max(FALLBACK_END_YEAR, span_max.year + 1)
    print(f"Bike-count data on disk spans {span_min.date()} .. {span_max.date()}")
else:
    START_YEAR, END_YEAR = FALLBACK_START_YEAR, FALLBACK_END_YEAR
    print("No bike-count data found on disk; using fallback year range.")

print(f"Using year range {START_YEAR}-{END_YEAR}")

Bike-count data on disk spans 2020-01-01 .. 2026-07-06
Using year range 2019-2027


## 2. Public holidays (NRW)

Rule-computed, no network call. Sanity-checked against a few well-known
fixed-date holidays.

In [3]:
public_holidays_df = public_holidays(
    START_YEAR, END_YEAR, subdiv=DEFAULT_PUBLIC_HOLIDAY_SUBDIV
)
print(f"{len(public_holidays_df)} public holidays across {START_YEAR}-{END_YEAR}")
public_holidays_df.head()

99 public holidays across 2019-2027


,date,name
0,2019-01-01,Neujahr
1,2019-04-19,Karfreitag
2,2019-04-22,Ostermontag
3,2019-05-01,Erster Mai
4,2019-05-30,Christi Himmelfahrt


In [4]:
# Known fixed-date holidays: confirm they land on the expected calendar
# dates for a recent, fully-covered year.
check_year = 2025
known = {
    f"{check_year}-01-01": "Neujahr",
    f"{check_year}-10-03": "Tag der Deutschen Einheit",
    f"{check_year}-12-25": "1. Weihnachtstag",
}
by_date = public_holidays_df.set_index(public_holidays_df["date"].dt.date.astype(str))["name"]
for iso_date, expected_name in known.items():
    found = by_date.get(iso_date)
    status = "OK" if found is not None else "MISSING"
    print(f"{status:7s} {iso_date}  expected~{expected_name!r:30s} found={found!r}")
    assert found is not None, f"{iso_date} ({expected_name}) not found in computed holidays"

OK      2025-01-01  expected~'Neujahr'                      found='Neujahr'
OK      2025-10-03  expected~'Tag der Deutschen Einheit'    found='Tag der Deutschen Einheit'
OK      2025-12-25  expected~'1. Weihnachtstag'             found='Erster Weihnachtstag'


## 3. Fetch and save NRW school holidays

Fetched from the OpenHolidays API, one request per calendar year (the
API rejects a `validFrom`/`validTo` span wider than 1095 days), then
schema-validated and cached as a single static CSV under
`data/raw/calendar/` - safe to re-run, always overwrites with the same
deterministic output rather than accumulating duplicates.

In [5]:
school_holidays_df = fetch_school_holidays(
    START_YEAR, END_YEAR, subdivision_code=DEFAULT_SUBDIVISION_CODE
)
school_holidays_path = save_school_holidays(school_holidays_df, CALENDAR_OUTPUT_DIR)
print(f"{len(school_holidays_df)} school-holiday periods -> {school_holidays_path}")
school_holidays_df.head(10)

45 school-holiday periods -> C:\Users\FloKI\Documents\Daten\muenster-bike-traffic-forecast\data\raw\calendar\school_holidays_nw.csv


,id,start_date,end_date,name,subdivision_code
0,a313a056-9c1f-4d0f-a1a4-88ba3fb92da3,2018-12-21,2019-01-04,Weihnachtsferien,DE-NW
1,998862f1-69e1-4c72-af0a-cdcdc78d8434,2019-04-15,2019-04-27,Osterferien,DE-NW
2,5a8ef93e-f2e8-404d-98b4-6f5160e2deac,2019-06-11,2019-06-11,Pfingstferien,DE-NW
3,7b241db2-7f38-4004-a204-9c43b41e5075,2019-07-15,2019-08-27,Sommerferien,DE-NW
4,37ccd1a0-9631-40ae-a6db-f15c193cbfb0,2019-10-14,2019-10-26,Herbstferien,DE-NW
5,78bbb200-36b2-4eb1-90bc-41ee649159f2,2019-12-23,2020-01-06,Weihnachtsferien,DE-NW
6,7da07aa0-f9f0-4d59-94cf-81958f5efffd,2020-04-06,2020-04-18,Osterferien,DE-NW
7,eae8403e-9a13-457f-a339-e871d22278f6,2020-06-02,2020-06-02,Pfingstferien,DE-NW
8,da87699c-6434-41dd-9a97-08c9f95073a3,2020-06-29,2020-08-11,Sommerferien,DE-NW
9,b38a47e0-9e8d-460c-b354-fb6be109ac75,2020-10-12,2020-10-24,Herbstferien,DE-NW


## 4. Sanity check: school-holiday day counts per year

Expands each `[start_date, end_date]` period into individual calendar
days (attributed to the year the day itself falls in - relevant for
periods like Weihnachtsferien that straddle a year boundary), then
reports the total number of school-holiday days per year. NRW typically
has on the order of ~65-75 school-holiday days per year, so this is a
plausibility check, not an exact spec.

In [6]:
expanded_days = pd.concat(
    [
        pd.DataFrame({"date": pd.date_range(row.start_date, row.end_date, freq="D")})
        for row in school_holidays_df.itertuples()
    ],
    ignore_index=True,
)
# A day could in principle be covered by two overlapping fetched periods;
# de-duplicate so it is only counted once.
expanded_days = expanded_days.drop_duplicates(subset="date")

days_per_year = expanded_days["date"].dt.year.value_counts().sort_index()
days_per_year.name = "school_holiday_days"
days_per_year

date
2018    11
2019    84
2020    88
2021    85
2022    86
2023    88
2024    85
2025    87
2026    88
2027    87
2028     8
Name: school_holiday_days, dtype: int64

## 5. Summary

- `public_holidays_df`: one row per NRW public holiday, columns `date`, `name` - computed in-memory, not persisted (cheap to
  recompute on demand).
- `school_holidays_df`: one row per NRW school-holiday period, saved to
  `data/raw/calendar/school_holidays_nw.csv`.

Both are keyed by calendar date and ready to be joined onto the 15-minute bike-count / hourly-weather timeline in a later
feature-engineering notebook.